In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Path to the merged model you copied from WSL
model_path = "../qwen_nutrition_merged_16bit"
# (If your notebook is in root folder, use: model_path = "./Unsloth_Chatmodel/qwen_nutrition_merged_16bit")

print("Loading tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,  # 16-bit half precision
    device_map="auto"           # Automatically uses your RTX 4060 GPU
)

print(f"Model successfully loaded onto: {model.device}")

Loading tokenizer and model...


[transformers] The tokenizer you are loading from '../qwen_nutrition_merged_16bit' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 777.56it/s]


Model successfully loaded onto: cuda:0


In [3]:
def chat_with_assistant(user_message, system_prompt=None, max_new_tokens=300):
    if system_prompt is None:
        system_prompt = (
            "You are a friendly and knowledgeable Indian nutrition and fitness assistant. "
            "You give practical, easy-to-follow advice about food, protein, workouts, "
            "Indian meals, and healthy habits. Keep your replies warm, natural and helpful."
        )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}
    ]

    # Apply the Qwen chat template
    text_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize and push to GPU
    model_inputs = tokenizer([text_prompt], return_tensors="pt").to(model.device)

    # Generate output
    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only the newly generated tokens
    input_len = model_inputs.input_ids.shape[1]
    response = tokenizer.decode(generated_ids[0][input_len:], skip_special_tokens=True)
    
    return response

In [4]:
query = "What are some high-protein vegetarian breakfast options in an Indian diet?"
reply = chat_with_assistant(query)

print("Assistant:\n", reply)

[transformers] Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Assistant:
 For a high-protein vegetarian breakfast that fits within your dietary preferences and goals, here are a few ideas: 1. **Paneer Paratha**: Made with whole wheat flour, paneer (cottage cheese), and spices, this is a great option for a savory meal. It’s rich in protein and carbs. 2. **Greek Yogurt with Berries**: A bowl of Greek yogurt topped with fresh berries can be refreshing and provide a good amount of protein. If you prefer something lighter, you could mix in some nuts or seeds for extra calories. 3. **Moong Dal Chilla**: These pancakes made from fermented chickpeas are nutritious and filling. They’re packed with protein and fiber. 4. **Egg Bhurji**: Scrambled eggs with spinach and tomatoes are a versatile way to add protein. You can season it with spices like cumin and turmeric. 5. **Oats with Milk and Nut Butter**: Cook oats with milk and top them with peanut butter or almond butter, which adds healthy fats and protein. Would any of these sound good to you? Or do you h

In [5]:
query = "I am a vegetarian trying to lose weight. Can I eat paneer daily?"
reply = chat_with_assistant(query)

print("Assistant:\n", reply)

[transformers] Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Assistant:
 Yes, you can definitely include paneer in your diet! It's a great option for weight loss since it's low in calories and high in protein. Here’s how you can incorporate it into your meals: 1. **Paneer Curry**: Try making a Paneer Tikka or adding it to a Dal with vegetables. This can be a delicious way to enhance the flavor while keeping your carbs in check. 2. **Paneer Sticks**: If you like them more crunchy, you could make a Paneer Bhurji with spices. 3. **Paneer with Rice**: Pairing Paneer with brown rice or millet gives you a balanced meal that supports both protein and fiber intake. 4. **Peanut Butter Toast**: Spread peanut butter on whole grain bread or toast instead of cheese. It's not only filling but also nutritious. 5. **Paneer Stir-Fry**: Sauté some paneer with veggies and tofu if you have non-veg options, then serve over a side of rice or quinoa. Just keep an eye on portion sizes since paneer is calorie-dense. How do these ideas sound? Do you have specific recipes

Chal CpU pe dekhte latency

In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import time

# 1. Path to your merged model folder on Windows
# Adjust path if your notebook is in a subfolder like 'Transformer_notebook'
model_path = "../qwen_nutrition_merged_16bit"
# If running inside Transformer_notebook folder, use:
# model_path = "../Unsloth_Chatmodel/qwen_nutrition_merged_16bit"

print("1. Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_path)

print("2. Loading Model on CPU (Float32 for CPU stability)...")
# Note: On CPU, float32 is the most stable and native format
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float32,   # Standard float32 precision for CPU
    device_map=None,             # Force disabling auto-GPU mapping
)
# Explicitly move model to CPU
model = model.to("cpu")
model.eval()

print("--> Model successfully loaded onto CPU!")

# 3. Define the CPU Chat function
def ask_assistant_cpu(user_query, max_tokens=250):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a friendly and knowledgeable Indian nutrition and fitness assistant. "
                "You give practical, easy-to-follow advice about food, protein, workouts, "
                "Indian meals, and healthy habits. Keep your replies warm, natural and helpful."
            ),
        },
        {"role": "user", "content": user_query},
    ]

    # Apply the Qwen chat template
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Convert tokens and place tensors explicitly on 'cpu'
    inputs = tokenizer([prompt_text], return_tensors="pt").to("cpu")

    start_time = time.time()
    
    # Generate output
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    duration = time.time() - start_time
    input_length = inputs.input_ids.shape[1]
    generated_tokens = output_ids[0][input_length:]
    
    # Calculate generation speed
    tokens_count = len(generated_tokens)
    speed = tokens_count / duration if duration > 0 else 0

    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    
    print(f"\n[Generated {tokens_count} tokens in {duration:.2f}s (~{speed:.1f} tokens/sec on CPU)]")
    return response

import time
# 4. Test Queries
print(f"start_time{time.time()}")
query_1 = "What are good vegetarian sources of protein in Indian food?"
print("\nUser:", query_1)
print("\nAssistant:\n", ask_assistant_cpu(query_1))
print(f"end_time{time.time()}")

# query_2 = "Can I drink chai every day while trying to lose weight?"
# print("\nUser:", query_2)
# print("\nAssistant:\n", ask_assistant_cpu(query_2))

1. Loading Tokenizer...


[transformers] The tokenizer you are loading from '../qwen_nutrition_merged_16bit' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


2. Loading Model on CPU (Float32 for CPU stability)...


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 571.43it/s]
[transformers] Both `max_new_tokens` (=250) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--> Model successfully loaded onto CPU!
start_time1791455976.728585

User: What are good vegetarian sources of protein in Indian food?

[Generated 250 tokens in 14.95s (~16.7 tokens/sec on CPU)]

Assistant:
 Good vegetarian sources of protein can vary based on your dietary preferences! Here are some options that fit your profile: 1. **Lentils**: Great for making dal or adding to curries. A cup of cooked lentils provides about 20 grams of protein. 2. **Chickpeas**: These can be used in salads, as a side dish, or even made into hummus. One cup of cooked chickpeas offers approximately 34 grams of protein. 3. **Quinoa**: This grain is high in protein and can be served with veggies or eaten as a complete protein alternative. A half-cup of cooked quinoa contains around 8 grams of protein. 4. **Tofu**: If you're okay with soy products, tofu has around 26 grams of protein per cooked ounce. It's versatile and can be added to many dishes. 5. **Greek Yogurt**: A serving of Greek yogurt (about 200